# シェル：取得・調査・圧縮

対応する本文は [shell.tex](../chapters/shell.tex) である。コードセルを上から順にShift+Enterで実行する。各問の説明にある値や図と、セルの下に表示される結果を比べる。ファイルを作る問では、その直前に保存先を用意する。

本文の命令は端末へ入力する。このNotebookのコードセルでは、Pythonの `subprocess.run()` で同じ命令を実行し、結果を表示する。最初は下に示す端末の命令と表示結果を対応させて読む。

## 問1：作業場所を作る

`mkdir -p` は親フォルダーも作成し、すでに存在する場合も続行する。`{raw,compressed,...}` は bash や zsh が展開する。`pwd` は現在位置を表示する。ここでは Notebook の起動位置を変えず、作業先を明示して操作する。

端末で入力する内容は次のとおりである。

```sh
mkdir -p earthquake_practice
cd earthquake_practice
mkdir -p raw compressed scripts figures results
pwd
ls
```

`mkdir` はフォルダーを作り、`cd` は移動する。最後に現在の場所と5個のフォルダー名を表示する。次のPythonセルでは `cwd=WORK` が開始場所を指定する。`capture_output=True` は命令の出力を受け取り、`result.stdout` に入れる。`check=True` は命令が失敗したときに知らせる。

### 保存先を用意する

この後のセルではファイルを作る。次のセルを実行して、`notebooks/_outputs/01_shell/` を用意する。表示された場所に結果が保存され、同名の実習用ファイルは再実行すると置き換わる。

Notebookは教材の `notebooks` フォルダー、または教材の一番上のフォルダーから実行する。`Path.cwd()` は現在のフォルダーを調べ、`mkdir()` は保存先を作る。

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "main.tex").is_file():
    raise FileNotFoundError("教材のフォルダーか notebooks フォルダーから実行してください。")
WORK = ROOT / "notebooks" / "_outputs" / "01_shell"
WORK.mkdir(parents=True, exist_ok=True)
print("Python", sys.version.split()[0])
print("保存先:", WORK.relative_to(ROOT))

Python 3.12.14
保存先: notebooks/_outputs/01_shell


この後のセルにある `assert 条件` は、結果を自動で確認する命令である。条件が違っていればエラーで止まり、条件を満たせば次へ進む。何も表示されなくても、エラーが出なければその確認は通っている。画面の数値や図も、説明に示した値と比べる。

In [2]:
import subprocess

commands = """mkdir -p earthquake_practice
cd earthquake_practice
mkdir -p raw compressed scripts figures results
pwd
ls
"""
result = subprocess.run(
    ["bash", "-c", commands], cwd=WORK,
    text=True, capture_output=True, check=True)
print(result.stdout)
PRACTICE = WORK / "earthquake_practice"
assert {p.name for p in PRACTICE.iterdir()} >= {"raw", "compressed", "scripts", "figures", "results"}

/Users/ikomae/src/python_test/tex/notebooks/_outputs/01_shell/earthquake_practice
compressed
figures
raw
results
scripts



## 問2：取得条件と行数を調べる

USGS の実カタログは更新されるため、固定の件数を解答とはしない。次のセルは通信せず、本文の人工 CSV を作る。`curl -fL` の `-f` は HTTP エラーを失敗とし、`-L` は転送先へ追従する。`-o` は出力先を指定する。

実データを取得する場合は、表示された `earthquake_practice` フォルダーへ端末で移動し、次を実行する。`count` は CSV 用の `format`、件数制限、並び順を付けずに問い合わせる。20,000件を超える場合は期間を分け、境界に重なる ID を除く。取得日時と検索条件も保存する。

```sh
curl -fL 'https://earthquake.usgs.gov/fdsnws/event/1/count?starttime=2024-01-01&endtime=2024-01-31T23:59:59.999999'   -o results/usgs_count.txt
curl -fL 'https://earthquake.usgs.gov/fdsnws/event/1/query?format=csv&starttime=2024-01-01&endtime=2024-01-31T23:59:59.999999&limit=20000&orderby=time-asc'   -o raw/usgs_global_2024_01.csv
```

`wc -l` は改行数を数える。今回の CSV ではヘッダーを含め4行、データは3件となる。一般の CSV にはフィールド内改行もあるため、行数から常にレコード数を決められるわけではない。`grep -i` は大文字と小文字を区別せずに行を検索する。

In [3]:
CSV_PATH = PRACTICE / "raw" / "usgs_example.csv"
CSV_PATH.write_text('time,latitude,longitude,mag,place\n2024-01-01T00:00:00.000Z,35.0,140.0,3.2,"Offshore, Japan"\n2024-01-01T00:05:00.000Z,36.0,141.0,,Japan\n2024-01-01T00:12:30.000Z,34.5,139.5,2.8,Tokyo\n', encoding="utf-8")
for args in (["wc", "-l", str(CSV_PATH)], ["head", "-n", "3", str(CSV_PATH)], ["tail", "-n", "3", str(CSV_PATH)], ["grep", "-i", "Japan", str(CSV_PATH)]):
    result = subprocess.run(args, text=True, capture_output=True, check=True)
    print(" ".join(args[:3]), "\n", result.stdout)
assert CSV_PATH.read_bytes().count(b"\n") == 4

wc -l /Users/ikomae/src/python_test/tex/notebooks/_outputs/01_shell/earthquake_practice/raw/usgs_example.csv 
        4 /Users/ikomae/src/python_test/tex/notebooks/_outputs/01_shell/earthquake_practice/raw/usgs_example.csv

head -n 3 
 time,latitude,longitude,mag,place
2024-01-01T00:00:00.000Z,35.0,140.0,3.2,"Offshore, Japan"
2024-01-01T00:05:00.000Z,36.0,141.0,,Japan

tail -n 3 
 2024-01-01T00:00:00.000Z,35.0,140.0,3.2,"Offshore, Japan"
2024-01-01T00:05:00.000Z,36.0,141.0,,Japan
2024-01-01T00:12:30.000Z,34.5,139.5,2.8,Tokyo

grep -i Japan 
 2024-01-01T00:00:00.000Z,35.0,140.0,3.2,"Offshore, Japan"
2024-01-01T00:05:00.000Z,36.0,141.0,,Japan



## 問3：圧縮と復元を照合する

gzip はデータを圧縮し、tar.gz は、複数のファイルと名前を一つにまとめたアーカイブを gzip で圧縮する。ZIP はファイル名なども保持する。容量比は「圧縮後 / 圧縮前」であり、小さい入力では管理情報のため1を超えることもある。先頭行や格納名の表示だけでは一致を確認できないので、復元した全バイトを比較する。下では標準ライブラリで同じ3形式を作り、復元先を分けて `cmp` も実行する。

In [4]:
import gzip
import tarfile
import zipfile

raw = CSV_PATH.read_bytes()
compressed = PRACTICE / "compressed"
paths = [compressed / "example.csv.gz", compressed / "example.tar.gz", compressed / "example.zip"]
with gzip.open(paths[0], "wb") as f:
    f.write(raw)
with tarfile.open(paths[1], "w:gz") as f:
    f.add(CSV_PATH, arcname="raw/usgs_example.csv")
with zipfile.ZipFile(paths[2], "w", compression=zipfile.ZIP_DEFLATED) as f:
    f.writestr("raw/usgs_example.csv", raw)
with gzip.open(paths[0], "rb") as f:
    restored_gzip = f.read()
with tarfile.open(paths[1], "r:gz") as f:
    restored_tar = f.extractfile("raw/usgs_example.csv").read()
with zipfile.ZipFile(paths[2]) as f:
    restored_zip = f.read("raw/usgs_example.csv")
for path, restored in zip(paths, [restored_gzip, restored_tar, restored_zip]):
    target = PRACTICE / "results" / path.suffix.lstrip(".") / path.stem / "restored.csv"
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_bytes(restored)
    assert restored == raw
    subprocess.run(["cmp", str(CSV_PATH), str(target)], check=True)
    print(path.name, path.stat().st_size, "bytes; 容量比 =", round(path.stat().st_size / len(raw), 3))

example.csv.gz 151 bytes; 容量比 = 0.834
example.tar.gz 315 bytes; 容量比 = 1.74
example.zip 259 bytes; 容量比 = 1.431


## 問4：URL を分解する

`https` はスキーム、`earthquake.usgs.gov` はホスト名、`usgs.gov` は登録ドメイン、`.gov` はトップレベルドメインである。パスは機能を、クエリは検索条件を指定する。`format` は出力形式、`starttime` と `endtime` は UTC の期間、`limit` は上限件数、`orderby=time-asc` は時刻の昇順を指定する。

[USGS の公式 API 仕様](https://earthquake.usgs.gov/fdsnws/event/1/)で各指定を確認できる。

In [5]:
from urllib.parse import parse_qs, urlsplit

url = "https://earthquake.usgs.gov/fdsnws/event/1/query?format=csv&starttime=2024-01-01&endtime=2024-01-31T23:59:59.999999&limit=20000&orderby=time-asc"
parts = urlsplit(url)
print(parts.scheme, parts.hostname, parts.path)
print(parse_qs(parts.query))
assert parse_qs(parts.query)["limit"] == ["20000"]

https earthquake.usgs.gov /fdsnws/event/1/query
{'format': ['csv'], 'starttime': ['2024-01-01'], 'endtime': ['2024-01-31T23:59:59.999999'], 'limit': ['20000'], 'orderby': ['time-asc']}
